# IP/GV target–program CRT

**Selected cNMF model:** `K=30`, density threshold `2.0`, 100 cNMF replicates.

The pilot metrics do not provide a strong reason to reject density threshold 2.0 at K=30:

- total explained variance is effectively unchanged;
- the fraction of programs with significant GO terms is the same;
- Reactome coverage is slightly higher at threshold 2.0;
- the mouse-associated fraction is the same;
- cross-mouse perturbation concordance is modestly lower but remains in the same range.

This notebook first compares CRT calibration across covariate sets using pseudo-targets made from non-targeting guides. It then optionally runs the full target–program analysis.

Used the **Python (programDE IP-GV)** kernel created by `setup_programDE_ipgv_mamba.sh`.


In [ ]:
import os

for variable in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")

os.environ.setdefault("PYTHONNOUSERSITE", "1")


In [ ]:
from __future__ import annotations

from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
import inspect
import json
import math
import re
import subprocess
import sys
import time
from typing import Any

import anndata as ad
from IPython.display import display
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import kstest, norm
from statsmodels.stats.multitest import multipletests

from src.sceptre import (
    build_ntc_group_inputs,
    compute_guide_set_null_pvals,
    crt_pvals_for_ntc_groups_ensemble,
    crt_pvals_for_ntc_groups_ensemble_skew,
    make_ntc_groups_ensemble,
    prepare_crt_inputs,
    run_all_genes_union_crt,
)
from src.visualization import qq_plot_ntc_pvals

try:
    from src.sceptre import limit_threading
    limit_threading()
except (ImportError, AttributeError):
    pass

print("Python:", sys.version.split()[0])
print("prepare_crt_inputs:", inspect.signature(prepare_crt_inputs))
print("run_all_genes_union_crt:", inspect.signature(run_all_genes_union_crt))


## Configuration

In [ ]:
# ---------------------------------------------------------------------
# Final selected cNMF model
# ---------------------------------------------------------------------
K = 30
DENSITY_THRESHOLD = 2.0

BASE = Path(
    "/oak/stanford/groups/engreitz/Users/tri/10xPerturbSeq/"
    "analysis/250507_ePAH1_NovaSeq/CRISPR_count/"
    "IP_GV_unified_sceptre_cnmf_no_celltype"
)

RESULTS_ROOT = BASE / "PerturbNMF" / "results"
EXPECTED_RUN_NAME = "IP_GV_skcnmf_K30_dt2_0_n100"
EXPECTED_H5MU = (
    RESULTS_ROOT
    / EXPECTED_RUN_NAME
    / "Inference"
    / "adata"
    / "cNMF_30_2_0.h5mu"
)

# Auto-discover only when the expected path is absent.
if EXPECTED_H5MU.exists():
    H5MU_PATH = EXPECTED_H5MU
else:
    candidates = sorted(
        RESULTS_ROOT.glob(
            "*K30*dt2_0*n100/Inference/adata/cNMF_30_2_0.h5mu"
        )
    )
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected {EXPECTED_H5MU}, and auto-discovery found "
            f"{len(candidates)} candidates: {candidates}"
        )
    H5MU_PATH = candidates[0]

RUN_ROOT = H5MU_PATH.parents[2]
CRT_ROOT = (
    RUN_ROOT
    / "Evaluation"
    / "30_2_0"
    / "CRT_notebook"
)
CALIBRATION_ROOT = CRT_ROOT / "calibration"
DISCOVERY_ROOT = CRT_ROOT / "discovery"

for directory in [CRT_ROOT, CALIBRATION_ROOT, DISCOVERY_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

CONTROL_TARGET_LABELS = ["non-targeting"]
CATEGORICAL_KEY = "sample"

# Calibration defaults
CALIBRATION_B = 1024
GUIDES_PER_PSEUDO_TARGET = 4
NTC_ENSEMBLES = 20
NTC_MAX_GROUPS = 7
NTC_FREQUENCY_BINS = 5
NTC_GROUP_SEED = 7
NTC_CRT_SEED = 23
CALIBRATION_COMPUTE_SKEW = True

# Full production run defaults
DISCOVERY_B = 5000
N_JOBS = min(16, os.cpu_count() or 1)

print("H5MU:", H5MU_PATH)
print("Run root:", RUN_ROOT)
print("CRT output:", CRT_ROOT)
print("n_jobs:", N_JOBS)


## Load and validate the final H5MU

In [ ]:
mdata = mu.read(H5MU_PATH)

print(mdata)
print("Modalities:", list(mdata.mod.keys()))

if "rna" not in mdata.mod or "cNMF" not in mdata.mod:
    raise KeyError("Expected RNA and cNMF modalities.")

rna = mdata["rna"]
programs = mdata["cNMF"]

if programs.n_vars != K:
    raise ValueError(
        f"Expected K={K} programs, found {programs.n_vars}."
    )
if rna.n_obs != programs.n_obs:
    raise ValueError("RNA and cNMF modalities have different cell counts.")
if not rna.obs_names.equals(programs.obs_names):
    raise ValueError("RNA and cNMF cell orders differ.")

print("RNA shape:", rna.shape)
print("Program shape:", programs.shape)


In [ ]:
def resolve_obs_series(key: str) -> pd.Series:
    """Resolve an obs column from cNMF first, then RNA."""
    if key in programs.obs:
        values = programs.obs[key]
    elif key in rna.obs:
        values = rna.obs.loc[programs.obs_names, key]
    else:
        raise KeyError(
            f"{key!r} is absent from both cNMF.obs and RNA.obs."
        )
    return pd.Series(
        values.to_numpy(),
        index=programs.obs_names,
        name=key,
    )


# Resolve guide metadata from cNMF first, with RNA fallback.
if "guide_assignment" in programs.obsm:
    guide_assignment = programs.obsm["guide_assignment"]
elif "guide_assignments" in programs.obsm:
    guide_assignment = programs.obsm["guide_assignments"]
elif "guide_assignment" in rna.obsm:
    guide_assignment = rna.obsm["guide_assignment"]
elif "guide_assignments" in rna.obsm:
    guide_assignment = rna.obsm["guide_assignments"]
else:
    raise KeyError("No guide-assignment matrix found.")

if sp.issparse(guide_assignment):
    guide_assignment = guide_assignment.tocsr().copy()
else:
    guide_assignment = sp.csr_matrix(np.asarray(guide_assignment))

guide_assignment.eliminate_zeros()
guide_assignment.data = np.ones(
    guide_assignment.nnz,
    dtype=np.int8,
)

guide_names = None
guide_targets = None
for modality in [programs, rna]:
    if guide_names is None and "guide_names" in modality.uns:
        guide_names = np.asarray(modality.uns["guide_names"]).astype(str)
    if guide_targets is None and "guide_targets" in modality.uns:
        guide_targets = np.asarray(
            modality.uns["guide_targets"]
        ).astype(str)

if guide_names is None or guide_targets is None:
    raise KeyError("guide_names or guide_targets are missing.")
if guide_assignment.shape != (programs.n_obs, len(guide_names)):
    raise ValueError(
        f"Guide assignment has shape {guide_assignment.shape}; "
        f"expected {(programs.n_obs, len(guide_names))}."
    )
if len(guide_targets) != len(guide_names):
    raise ValueError("guide_names and guide_targets lengths differ.")

usage = np.asarray(programs.X, dtype=float)
if usage.shape != (programs.n_obs, K):
    raise ValueError("Unexpected program-usage shape.")
if not np.isfinite(usage).all() or np.any(usage < 0):
    raise ValueError("Program usage contains invalid values.")

usage = np.maximum(usage, 1e-8)
usage /= usage.sum(axis=1, keepdims=True)

sample = resolve_obs_series(CATEGORICAL_KEY).astype(str)
sample_counts = sample.value_counts()

target_counts = (
    pd.Series(guide_targets, name="target")
    .value_counts()
    .rename("n_guides")
)

assigned_per_cell = np.asarray(
    guide_assignment.sum(axis=1)
).ravel()

print("Sample counts:")
display(sample_counts.to_frame())

print("Guide counts by target:")
display(target_counts.to_frame())

print(
    "Assigned guides/cell:",
    pd.Series(assigned_per_cell).describe().to_dict(),
)

if not set(CONTROL_TARGET_LABELS).intersection(set(guide_targets)):
    raise ValueError(
        f"Control labels {CONTROL_TARGET_LABELS} do not occur in guide_targets."
    )


In [ ]:
available_obs = sorted(
    set(programs.obs.columns).union(rna.obs.columns)
)
print("Available obs columns:")
print(available_obs)

candidate_keys = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
    "n_assigned_guides",
    "total_guide_umis",
    "assigned_guide_umis",
]

summary_rows = []
for key in candidate_keys:
    if key in available_obs:
        values = pd.to_numeric(
            resolve_obs_series(key),
            errors="coerce",
        )
        summary_rows.append(
            {
                "key": key,
                "missing": int(values.isna().sum()),
                "n_unique": int(values.nunique(dropna=True)),
                "min": float(values.min()),
                "median": float(values.median()),
                "max": float(values.max()),
            }
        )

display(pd.DataFrame(summary_rows))


## Candidate covariate sets

The CRT is run separately within each mouse, so `sample` is not included as a within-mouse covariate.

The initial variants are intentionally limited:

- **none**: intercept only;
- **rna_qc**: RNA depth, detected genes, mitochondrial and ribosomal percentages;
- **rna_qc_guides**: RNA QC plus assigned-guide number and guide UMI burden;
- **rna_qc_guides_hb**: adds hemoglobin percentage when available.

Avoid including redundant columns such as both total and assigned guide UMIs in the same model. Prefer the simplest set that gives acceptable calibration.


In [ ]:
def existing(keys: list[str]) -> list[str]:
    return [key for key in keys if key in available_obs]


COVARIATE_VARIANTS = OrderedDict(
    {
        "none": {
            "covariates": [],
            "log_covariates": [],
        },
        "rna_qc": {
            "covariates": existing(
                ["pct_counts_mt", "pct_counts_ribo"]
            ),
            "log_covariates": existing(
                ["total_counts", "n_genes_by_counts"]
            ),
        },
        "rna_qc_guides": {
            "covariates": existing(
                ["pct_counts_mt", "pct_counts_ribo"]
            ),
            "log_covariates": existing(
                [
                    "total_counts",
                    "n_genes_by_counts",
                    "n_assigned_guides",
                    "total_guide_umis",
                ]
            ),
        },
        "rna_qc_guides_hb": {
            "covariates": existing(
                [
                    "pct_counts_mt",
                    "pct_counts_ribo",
                    "pct_counts_hb",
                ]
            ),
            "log_covariates": existing(
                [
                    "total_counts",
                    "n_genes_by_counts",
                    "n_assigned_guides",
                    "total_guide_umis",
                ]
            ),
        },
    }
)

display(
    pd.DataFrame(
        [
            {
                "variant": name,
                "covariates": ", ".join(spec["covariates"]),
                "log_covariates": ", ".join(
                    spec["log_covariates"]
                ),
            }
            for name, spec in COVARIATE_VARIANTS.items()
        ]
    )
)


## CRT helper functions

In [ ]:
def safe_name(value: object) -> str:
    return (
        re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")
        or "condition"
    )


def build_crt_adata(
    covariates: list[str],
    log_covariates: list[str],
) -> ad.AnnData:
    adata = programs.copy()
    adata.obsm["cnmf_usage"] = usage.copy()
    adata.obsm["guide_assignment"] = guide_assignment.copy()
    adata.uns["program_names"] = list(adata.var_names)
    adata.uns["guide_names"] = guide_names.tolist()
    adata.uns["guide2gene"] = dict(
        zip(guide_names.tolist(), guide_targets.tolist())
    )

    if CATEGORICAL_KEY not in adata.obs:
        adata.obs[CATEGORICAL_KEY] = resolve_obs_series(
            CATEGORICAL_KEY
        ).to_numpy()

    adata.obs["guides_per_cell"] = assigned_per_cell

    covar = pd.DataFrame(index=adata.obs_names)

    for key in covariates:
        values = pd.to_numeric(
            resolve_obs_series(key),
            errors="coerce",
        )
        if values.isna().any():
            raise ValueError(
                f"Covariate {key!r} has missing/non-numeric values."
            )
        covar[key] = values

    for key in log_covariates:
        values = pd.to_numeric(
            resolve_obs_series(key),
            errors="coerce",
        )
        if values.isna().any():
            raise ValueError(
                f"Log covariate {key!r} has missing/non-numeric values."
            )
        if (values < 0).any():
            raise ValueError(
                f"Log covariate {key!r} contains negative values."
            )
        covar[f"log_{key}"] = np.log1p(values)

    # Remove globally constant covariates. A second condition-specific
    # check is performed below.
    keep = [
        column
        for column in covar.columns
        if covar[column].nunique(dropna=False) > 1
    ]
    adata.obsm["covar"] = covar[keep]
    return adata


def condition_subset(
    adata: ad.AnnData,
    condition: str,
) -> ad.AnnData:
    mask = (
        adata.obs[CATEGORICAL_KEY].astype(str)
        == str(condition)
    )
    subset = adata[mask].copy()

    covar = subset.obsm["covar"]
    if not isinstance(covar, pd.DataFrame):
        covar = pd.DataFrame(
            np.asarray(covar),
            index=subset.obs_names,
        )

    keep = [
        column
        for column in covar.columns
        if covar[column].nunique(dropna=False) > 1
    ]
    subset.obsm["covar"] = covar[keep]
    return subset


def prepare_inputs(adata: ad.AnnData):
    kwargs = dict(
        adata=adata,
        usage_key="cnmf_usage",
        covar_key="covar",
        guide_assignment_key="guide_assignment",
        guide2gene_key="guide2gene",
    )
    signature = inspect.signature(prepare_crt_inputs)

    # Treat numeric covariates as continuous even when they have <=20
    # distinct values, such as guides per cell.
    if "numeric_as_category_threshold" in signature.parameters:
        kwargs["numeric_as_category_threshold"] = None

    return prepare_crt_inputs(**kwargs)


def flatten_ntc_frames(frames: dict) -> np.ndarray:
    arrays = []
    for frame in frames.values():
        values = frame.to_numpy(dtype=float, copy=False).ravel()
        arrays.append(values[np.isfinite(values)])
    if not arrays:
        return np.empty(0, dtype=float)
    return np.concatenate(arrays)


def qq_xy(pvalues: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(pvalues, dtype=float)
    values = values[np.isfinite(values)]
    values = np.clip(values, 1e-300, 1.0)
    values.sort()
    n = values.size
    if n == 0:
        return np.empty(0), np.empty(0)
    expected = (np.arange(1, n + 1) - 0.5) / n
    return -np.log10(expected), -np.log10(values)


def calibration_metrics(pvalues: np.ndarray) -> dict[str, float]:
    values = np.asarray(pvalues, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return {"n_pvalues": 0}

    ks = kstest(values, "uniform")
    result = {
        "n_pvalues": int(values.size),
        "mean_p": float(values.mean()),
        "median_p": float(np.median(values)),
        "ks_stat": float(ks.statistic),
        "ks_pvalue": float(ks.pvalue),
    }

    for threshold in [0.05, 0.01, 0.001]:
        fraction = float(np.mean(values <= threshold))
        result[f"fraction_p_le_{threshold:g}"] = fraction
        result[f"inflation_ratio_{threshold:g}"] = (
            fraction / threshold
        )

    return result


def save_ntc_frames(
    frames: dict,
    path: Path,
) -> None:
    rows = []
    for ensemble, frame in frames.items():
        current = frame.copy()
        current.index.name = "group_id"
        current = current.reset_index()
        current.insert(0, "ensemble", ensemble)
        rows.append(current)

    if rows:
        pd.concat(rows, ignore_index=True).to_csv(
            path,
            sep="\t",
            index=False,
        )


@dataclass
class CalibrationResult:
    variant: str
    condition: str
    raw_pvalues: np.ndarray
    skew_pvalues: np.ndarray
    crt_null_pvalues: np.ndarray
    output_dir: Path


def plot_calibration_result(
    result: CalibrationResult,
    title: str | None = None,
):
    figure, axis = plt.subplots(figsize=(7, 6))

    plotted = []
    for label, values, style in [
        ("NTC pseudo-target empirical", result.raw_pvalues, "-"),
        ("NTC pseudo-target skew", result.skew_pvalues, "--"),
        ("CRT-null leave-one-out", result.crt_null_pvalues, ":"),
    ]:
        x, y = qq_xy(values)
        if x.size:
            axis.plot(x, y, style, label=label)
            plotted.extend([x.max(), y.max()])

    limit = max([1.0] + [float(value) for value in plotted])
    axis.plot([0, limit], [0, limit], label="y=x")
    axis.set_xlim(0, limit)
    axis.set_ylim(0, limit)
    axis.set_xlabel("Expected -log10(p)")
    axis.set_ylabel("Observed -log10(p)")
    axis.set_title(
        title
        or f"{result.variant}: {result.condition}"
    )
    axis.legend(frameon=False)
    figure.tight_layout()
    return figure, axis


In [ ]:
def run_ntc_calibration(
    variant_name: str,
    condition: str,
    *,
    B: int = CALIBRATION_B,
    compute_skew: bool = CALIBRATION_COMPUTE_SKEW,
    overwrite: bool = False,
) -> CalibrationResult:
    specification = COVARIATE_VARIANTS[variant_name]
    output_dir = (
        CALIBRATION_ROOT
        / variant_name
        / safe_name(condition)
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_path = output_dir / "ntc_empirical_pvalues.npy"
    skew_path = output_dir / "ntc_skew_pvalues.npy"
    null_path = output_dir / "crt_null_pvalues.npy"

    if (
        not overwrite
        and raw_path.exists()
        and null_path.exists()
        and (not compute_skew or skew_path.exists())
    ):
        return CalibrationResult(
            variant=variant_name,
            condition=condition,
            raw_pvalues=np.load(raw_path),
            skew_pvalues=(
                np.load(skew_path)
                if skew_path.exists()
                else np.empty(0)
            ),
            crt_null_pvalues=np.load(null_path),
            output_dir=output_dir,
        )

    adata = build_crt_adata(
        specification["covariates"],
        specification["log_covariates"],
    )
    adata_condition = condition_subset(adata, condition)
    print(
        f"{variant_name} / {condition}: "
        f"{adata_condition.n_obs:,} cells, "
        f"{adata_condition.obsm['covar'].shape[1]} covariates"
    )

    inputs = prepare_inputs(adata_condition)

    (
        ntc_guides,
        guide_frequency,
        guide_to_bin,
        real_gene_bin_signatures,
    ) = build_ntc_group_inputs(
        inputs=inputs,
        ntc_label=CONTROL_TARGET_LABELS,
        group_size=GUIDES_PER_PSEUDO_TARGET,
        n_bins=NTC_FREQUENCY_BINS,
    )

    groups = make_ntc_groups_ensemble(
        ntc_guides=ntc_guides,
        ntc_freq=guide_frequency,
        real_gene_bin_sigs=real_gene_bin_signatures,
        guide_to_bin=guide_to_bin,
        n_ensemble=NTC_ENSEMBLES,
        seed0=NTC_GROUP_SEED,
        group_size=GUIDES_PER_PSEUDO_TARGET,
        max_groups=NTC_MAX_GROUPS,
    )

    empirical_frames = crt_pvals_for_ntc_groups_ensemble(
        inputs=inputs,
        ntc_groups_ens=groups,
        B=B,
        seed0=NTC_CRT_SEED,
    )
    empirical = flatten_ntc_frames(empirical_frames)

    if compute_skew:
        skew_frames = crt_pvals_for_ntc_groups_ensemble_skew(
            inputs=inputs,
            ntc_groups_ens=groups,
            B=B,
            seed0=NTC_CRT_SEED,
        )
        skew = flatten_ntc_frames(skew_frames)
    else:
        skew_frames = {}
        skew = np.empty(0, dtype=float)

    guide_to_column = {
        guide: index
        for index, guide in enumerate(inputs.guide_names)
    }
    null_chunks = []
    for ensemble_groups in groups:
        for guides in ensemble_groups.values():
            null_chunks.append(
                compute_guide_set_null_pvals(
                    guide_idx=[
                        guide_to_column[guide]
                        for guide in guides
                    ],
                    inputs=inputs,
                    B=B,
                ).ravel()
            )
    null_pvalues = (
        np.concatenate(null_chunks)
        if null_chunks
        else np.empty(0, dtype=float)
    )

    np.save(raw_path, empirical)
    np.save(skew_path, skew)
    np.save(null_path, null_pvalues)

    save_ntc_frames(
        empirical_frames,
        output_dir / "ntc_empirical_pvalues_wide.tsv",
    )
    if skew_frames:
        save_ntc_frames(
            skew_frames,
            output_dir / "ntc_skew_pvalues_wide.tsv",
        )

    config = {
        "variant": variant_name,
        "condition": condition,
        "B": B,
        "covariates": specification["covariates"],
        "log_covariates": specification["log_covariates"],
        "guides_per_pseudo_target": GUIDES_PER_PSEUDO_TARGET,
        "ntc_ensembles": NTC_ENSEMBLES,
        "ntc_max_groups": NTC_MAX_GROUPS,
        "ntc_frequency_bins": NTC_FREQUENCY_BINS,
        "ntc_group_seed": NTC_GROUP_SEED,
        "ntc_crt_seed": NTC_CRT_SEED,
    }
    (output_dir / "config.json").write_text(
        json.dumps(config, indent=2) + "\n"
    )

    result = CalibrationResult(
        variant=variant_name,
        condition=condition,
        raw_pvalues=empirical,
        skew_pvalues=skew,
        crt_null_pvalues=null_pvalues,
        output_dir=output_dir,
    )
    figure, _ = plot_calibration_result(result)
    figure.savefig(
        output_dir / "calibration_qq.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(figure)
    return result


## Run the calibration grid

Set `RUN_CALIBRATION_GRID = True` and run this cell. Results are cached on disk, so rerunning the cell does not recompute completed variants unless `OVERWRITE_CALIBRATION = True`.


In [ ]:
RUN_CALIBRATION_GRID = True
OVERWRITE_CALIBRATION = False

conditions = sample_counts.index.astype(str).tolist()
calibration_results: dict[tuple[str, str], CalibrationResult] = {}

if RUN_CALIBRATION_GRID:
    started = time.time()

    for variant_name in COVARIATE_VARIANTS:
        for condition in conditions:
            result = run_ntc_calibration(
                variant_name,
                condition,
                overwrite=OVERWRITE_CALIBRATION,
            )
            calibration_results[
                (variant_name, condition)
            ] = result

            figure, _ = plot_calibration_result(result)
            display(figure)
            plt.close(figure)

    print(
        "Elapsed minutes:",
        round((time.time() - started) / 60, 2),
    )
else:
    print(
        "Calibration is disabled. Set RUN_CALIBRATION_GRID=True "
        "when ready."
    )


## Summarize and compare calibration

In [ ]:
# Load all completed calibration results from disk.
calibration_results = {}

for variant_name in COVARIATE_VARIANTS:
    for condition in conditions:
        directory = (
            CALIBRATION_ROOT
            / variant_name
            / safe_name(condition)
        )
        raw_path = directory / "ntc_empirical_pvalues.npy"
        skew_path = directory / "ntc_skew_pvalues.npy"
        null_path = directory / "crt_null_pvalues.npy"

        if raw_path.exists() and null_path.exists():
            calibration_results[
                (variant_name, condition)
            ] = CalibrationResult(
                variant=variant_name,
                condition=condition,
                raw_pvalues=np.load(raw_path),
                skew_pvalues=(
                    np.load(skew_path)
                    if skew_path.exists()
                    else np.empty(0)
                ),
                crt_null_pvalues=np.load(null_path),
                output_dir=directory,
            )

rows = []
for (variant_name, condition), result in calibration_results.items():
    for pvalue_type, values in [
        ("empirical", result.raw_pvalues),
        ("skew", result.skew_pvalues),
        ("crt_null", result.crt_null_pvalues),
    ]:
        if values.size == 0:
            continue
        row = {
            "variant": variant_name,
            "condition": condition,
            "pvalue_type": pvalue_type,
        }
        row.update(calibration_metrics(values))
        rows.append(row)

calibration_summary = pd.DataFrame(rows)

if calibration_summary.empty:
    print(
        "No calibration results found. Run the calibration grid first."
    )
else:
    # Distance is only a sorting aid, not an automatic selection rule.
    calibration_summary["calibration_distance"] = (
        calibration_summary["ks_stat"]
        + (
            calibration_summary["inflation_ratio_0.05"] - 1
        ).abs()
        + (
            calibration_summary["inflation_ratio_0.01"] - 1
        ).abs()
    )

    calibration_summary.to_csv(
        CALIBRATION_ROOT / "calibration_summary.tsv",
        sep="\t",
        index=False,
    )

    display(
        calibration_summary.sort_values(
            [
                "pvalue_type",
                "condition",
                "calibration_distance",
            ]
        )
    )


In [ ]:
# Overlay candidate variants for each mouse and p-value family.
if not calibration_results:
    print("No calibration results to plot.")
else:
    for condition in conditions:
        for pvalue_type in ["empirical", "skew"]:
            figure, axis = plt.subplots(figsize=(7, 6))
            maxima = [1.0]

            for variant_name in COVARIATE_VARIANTS:
                key = (variant_name, condition)
                if key not in calibration_results:
                    continue

                result = calibration_results[key]
                values = (
                    result.raw_pvalues
                    if pvalue_type == "empirical"
                    else result.skew_pvalues
                )
                x, y = qq_xy(values)
                if x.size == 0:
                    continue

                axis.plot(x, y, label=variant_name)
                maxima.extend([float(x.max()), float(y.max())])

            limit = max(maxima)
            axis.plot([0, limit], [0, limit], label="y=x")
            axis.set_xlim(0, limit)
            axis.set_ylim(0, limit)
            axis.set_xlabel("Expected -log10(p)")
            axis.set_ylabel("Observed -log10(p)")
            axis.set_title(
                f"{condition}: {pvalue_type} NTC calibration"
            )
            axis.legend(frameon=False)
            figure.tight_layout()

            figure.savefig(
                CALIBRATION_ROOT
                / f"compare_{safe_name(condition)}_{pvalue_type}.png",
                dpi=180,
                bbox_inches="tight",
            )
            display(figure)
            plt.close(figure)


## Choose the final calibration specification

Guidelines:

- `fraction_p_le_0.05` should be near 0.05;
- `fraction_p_le_0.01` should be near 0.01;
- inflation ratios substantially above 1 indicate anti-conservative p-values;
- values substantially below 1 indicate conservative/deflated p-values;
- prefer a simpler covariate set when calibration is similar;
- verify calibration in **both mice**.

The default below is a starting point, not an automatic conclusion.


In [ ]:
SELECTED_VARIANT = "rna_qc_guides"
PRIMARY_PVALUE = "skew"  # "skew" or "empirical"

if SELECTED_VARIANT not in COVARIATE_VARIANTS:
    raise KeyError(SELECTED_VARIANT)
if PRIMARY_PVALUE not in {"skew", "empirical"}:
    raise ValueError(PRIMARY_PVALUE)

print("Selected variant:", SELECTED_VARIANT)
print("Specification:", COVARIATE_VARIANTS[SELECTED_VARIANT])
print("Primary p-value:", PRIMARY_PVALUE)

if not calibration_summary.empty:
    display(
        calibration_summary.loc[
            calibration_summary["variant"]
            == SELECTED_VARIANT
        ].sort_values(["condition", "pvalue_type"])
    )


## Full target–program CRT

In [ ]:
def long_results_table(
    out: dict[str, Any],
    *,
    condition: str,
    primary_pvalue: str,
) -> pd.DataFrame:
    empirical = out["pvals_raw_df"]

    skew = out.get("pvals_df")
    if skew is None:
        skew = out.get("pvals_skew_df")

    beta = out["betas_df"]

    def melt(frame: pd.DataFrame, value_name: str):
        index_name = frame.index.name or "target_name"
        current = frame.copy()
        current.index.name = "target_name"
        return (
            current.reset_index()
            .melt(
                id_vars="target_name",
                var_name="program_name",
                value_name=value_name,
            )
        )

    table = melt(empirical, "p_empirical").merge(
        melt(beta, "beta_clr"),
        on=["target_name", "program_name"],
        how="inner",
    )

    if skew is not None:
        table = table.merge(
            melt(skew, "p_skew"),
            on=["target_name", "program_name"],
            how="left",
        )
    else:
        table["p_skew"] = np.nan

    table["q_empirical_BH"] = multipletests(
        table["p_empirical"].fillna(1.0),
        method="fdr_bh",
    )[1]

    if table["p_skew"].notna().any():
        table["q_skew_BH"] = multipletests(
            table["p_skew"].fillna(1.0),
            method="fdr_bh",
        )[1]
    else:
        table["q_skew_BH"] = np.nan

    # beta_clr is a CLR log-ratio effect, not an ordinary expression log2FC.
    table["approx_relative_log2fc"] = (
        (K / (K - 1))
        * table["beta_clr"]
        / np.log(2)
    )

    table["condition"] = condition
    table["primary_pvalue_type"] = primary_pvalue

    if primary_pvalue == "skew":
        table["primary_pvalue"] = table["p_skew"]
        table["primary_q_BH"] = table["q_skew_BH"]
    else:
        table["primary_pvalue"] = table["p_empirical"]
        table["primary_q_BH"] = table["q_empirical_BH"]

    return table


def signed_stouffer_two_mouse(
    table_a: pd.DataFrame,
    table_b: pd.DataFrame,
) -> pd.DataFrame:
    key = ["target_name", "program_name"]

    left = table_a[
        key + ["beta_clr", "primary_pvalue", "primary_q_BH"]
    ].rename(
        columns={
            "beta_clr": "beta_IP_GFP",
            "primary_pvalue": "p_IP_GFP",
            "primary_q_BH": "q_IP_GFP",
        }
    )
    right = table_b[
        key + ["beta_clr", "primary_pvalue", "primary_q_BH"]
    ].rename(
        columns={
            "beta_clr": "beta_GV_GFP",
            "primary_pvalue": "p_GV_GFP",
            "primary_q_BH": "q_GV_GFP",
        }
    )

    merged = left.merge(right, on=key, how="inner")
    merged["same_sign"] = (
        np.sign(merged["beta_IP_GFP"])
        == np.sign(merged["beta_GV_GFP"])
    )

    p_ip = merged["p_IP_GFP"].clip(1e-300, 1.0)
    p_gv = merged["p_GV_GFP"].clip(1e-300, 1.0)

    z_ip = np.sign(merged["beta_IP_GFP"]) * norm.isf(
        p_ip / 2
    )
    z_gv = np.sign(merged["beta_GV_GFP"]) * norm.isf(
        p_gv / 2
    )
    z_combined = (z_ip + z_gv) / math.sqrt(2)

    merged["signed_stouffer_z"] = z_combined
    merged["signed_stouffer_p"] = (
        2 * norm.sf(np.abs(z_combined))
    )
    merged["signed_stouffer_q_BH"] = multipletests(
        merged["signed_stouffer_p"],
        method="fdr_bh",
    )[1]
    return merged


def run_full_crt_for_condition(
    condition: str,
    *,
    variant_name: str,
    B: int,
    primary_pvalue: str,
    overwrite: bool = False,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    specification = COVARIATE_VARIANTS[variant_name]
    output_dir = (
        DISCOVERY_ROOT
        / variant_name
        / safe_name(condition)
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    table_path = output_dir / "target_program_CRT.tsv"
    if table_path.exists() and not overwrite:
        return pd.read_csv(table_path, sep="\t"), {}

    adata = build_crt_adata(
        specification["covariates"],
        specification["log_covariates"],
    )
    adata_condition = condition_subset(adata, condition)
    inputs = prepare_inputs(adata_condition)

    print(
        f"Running full CRT for {condition}: "
        f"{adata_condition.n_obs:,} cells, B={B:,}"
    )

    out = run_all_genes_union_crt(
        inputs=inputs,
        B=B,
        n_jobs=N_JOBS,
        calibrate_skew_normal=True,
        return_raw_pvals=True,
        return_skew_normal=True,
    )

    # Recreate matched NTC pseudo-targets for the final QQ plot.
    (
        ntc_guides,
        guide_frequency,
        guide_to_bin,
        real_gene_bin_signatures,
    ) = build_ntc_group_inputs(
        inputs=inputs,
        ntc_label=CONTROL_TARGET_LABELS,
        group_size=GUIDES_PER_PSEUDO_TARGET,
        n_bins=NTC_FREQUENCY_BINS,
    )
    groups = make_ntc_groups_ensemble(
        ntc_guides=ntc_guides,
        ntc_freq=guide_frequency,
        real_gene_bin_sigs=real_gene_bin_signatures,
        guide_to_bin=guide_to_bin,
        n_ensemble=NTC_ENSEMBLES,
        seed0=NTC_GROUP_SEED,
        group_size=GUIDES_PER_PSEUDO_TARGET,
        max_groups=NTC_MAX_GROUPS,
    )

    ntc_empirical = crt_pvals_for_ntc_groups_ensemble(
        inputs=inputs,
        ntc_groups_ens=groups,
        B=B,
        seed0=NTC_CRT_SEED,
    )
    ntc_skew = crt_pvals_for_ntc_groups_ensemble_skew(
        inputs=inputs,
        ntc_groups_ens=groups,
        B=B,
        seed0=NTC_CRT_SEED,
    )

    guide_to_column = {
        guide: index
        for index, guide in enumerate(inputs.guide_names)
    }
    null_pvalues = np.concatenate(
        [
            compute_guide_set_null_pvals(
                guide_idx=[
                    guide_to_column[guide]
                    for guide in guides
                ],
                inputs=inputs,
                B=B,
            ).ravel()
            for ensemble_groups in groups
            for guides in ensemble_groups.values()
        ]
    )

    skew_frame = out.get("pvals_df")
    if skew_frame is None:
        skew_frame = out.get("pvals_skew_df")

    axis = qq_plot_ntc_pvals(
        pvals_raw_df=out["pvals_raw_df"],
        guide2gene=adata_condition.uns["guide2gene"],
        ntc_genes=CONTROL_TARGET_LABELS,
        pvals_skew_df=skew_frame,
        null_pvals=null_pvalues,
        ntc_group_pvals_ens=ntc_empirical,
        ntc_group_pvals_skew_ens=ntc_skew,
        show_ntc_ensemble_band=True,
        show_all_pvals=True,
        title=f"K30, dt2.0 CRT calibration: {condition}",
    )
    figure = axis.figure
    figure.tight_layout()
    figure.savefig(
        output_dir / "CRT_calibration_QQ.png",
        dpi=180,
        bbox_inches="tight",
    )
    display(figure)
    plt.close(figure)

    table = long_results_table(
        out,
        condition=condition,
        primary_pvalue=primary_pvalue,
    )
    table.to_csv(table_path, sep="\t", index=False)

    metadata = {
        "K": K,
        "density_threshold": DENSITY_THRESHOLD,
        "condition": condition,
        "variant": variant_name,
        "covariates": specification["covariates"],
        "log_covariates": specification["log_covariates"],
        "B": B,
        "n_jobs": N_JOBS,
        "primary_pvalue": primary_pvalue,
        "effect_scale": "CLR log-ratio beta",
    }
    (output_dir / "config.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )
    return table, out


### Execute the full run

Set `RUN_FULL_DISCOVERY = True` only after reviewing the calibration grid and setting `SELECTED_VARIANT` and `PRIMARY_PVALUE`.

The production default is 5,000 permutations. This can be computationally intensive in an interactive Jupyter session; use a sufficiently large Sherlock Jupyter allocation.


In [ ]:
RUN_FULL_DISCOVERY = True
OVERWRITE_DISCOVERY = False

discovery_tables: dict[str, pd.DataFrame] = {}

if RUN_FULL_DISCOVERY:
    started = time.time()

    for condition in conditions:
        table, _ = run_full_crt_for_condition(
            condition,
            variant_name=SELECTED_VARIANT,
            B=DISCOVERY_B,
            primary_pvalue=PRIMARY_PVALUE,
            overwrite=OVERWRITE_DISCOVERY,
        )
        discovery_tables[condition] = table
        display(
            table.sort_values("primary_q_BH").head(20)
        )

    print(
        "Elapsed minutes:",
        round((time.time() - started) / 60, 2),
    )
else:
    print(
        "Full discovery is disabled. Set RUN_FULL_DISCOVERY=True "
        "after selecting the calibration specification."
    )


## Cross-mouse summary

In [ ]:
# Load completed per-mouse tables from disk.
discovery_tables = {}

for condition in conditions:
    path = (
        DISCOVERY_ROOT
        / SELECTED_VARIANT
        / safe_name(condition)
        / "target_program_CRT.tsv"
    )
    if path.exists():
        discovery_tables[condition] = pd.read_csv(
            path,
            sep="\t",
        )

if {"IP_GFP", "GV_GFP"}.issubset(discovery_tables):
    cross_mouse = signed_stouffer_two_mouse(
        discovery_tables["IP_GFP"],
        discovery_tables["GV_GFP"],
    )
    cross_mouse.to_csv(
        DISCOVERY_ROOT
        / SELECTED_VARIANT
        / "cross_mouse_target_program_summary.tsv",
        sep="\t",
        index=False,
    )

    display(
        cross_mouse.sort_values(
            "signed_stouffer_q_BH"
        ).head(30)
    )

    print(
        "Same-sign fraction:",
        float(cross_mouse["same_sign"].mean()),
    )
else:
    print(
        "Both per-mouse discovery tables are required before "
        "creating the cross-mouse summary."
    )


## Interpretation notes

- The CRT coefficient is a **CLR log-ratio effect**, not a conventional gene-expression log2 fold change.
- Empirical and skew-calibrated p-values are both retained.
- BH correction is applied within each mouse across all target–program tests.
- The signed-Stouffer table is an exploratory two-mouse summary. With only two biological replicates, it should not be treated as a substitute for a larger replicate-level analysis.
- Because this is a high-MOI experiment, inspect NTC calibration, guide burden, and cross-mouse direction consistency before prioritizing associations.
